In [13]:
import os
import glob
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
from PIL import Image
from tqdm import tqdm
Image.MAX_IMAGE_PIXELS = None  # 取消限制

# ---------------------------
# 读取蛋白编码基因列表（用于过滤）
# ---------------------------
df_gene = pd.read_csv("./database/protein_coding_hg38.txt", header=None, names=["id", "name"])
df_gene = df_gene.dropna(subset=["name"])
valid_genes = set(df_gene["name"].str.upper())

# ---------------------------
# 读取鼠-人映射表
# ---------------------------
df_map = pd.read_csv("./database/mouse_human_map.txt")  # 这个是你给的 txt
df_map = df_map.dropna(subset=["Mouse gene name", "Gene name"])
mouse2human = dict(
    zip(
        df_map["Mouse gene name"].str.lower(),  # 小鼠基因名转小写
        df_map["Gene name"].str.upper()         # 人类基因名全大写
    )
)

# ---------------------------
# tokenize 函数（按表达值高低排序，生成 top_k token）
# ---------------------------
def tokenize_spots(ad_ge, top_k=2048, label_mode="threshold", valid_gene_names=None):
    assert label_mode in ["percentile", "zscore", "threshold"]

    X = ad_ge.X  # shape: (n_spots, n_genes)
    gene_names = np.array([g.upper() for g in ad_ge.var_names])  # 保证大写

    # 过滤掉不在 valid_gene_names 的基因
    if valid_gene_names is not None:
        valid_mask = np.isin(gene_names, list(valid_gene_names))
        X = X[:, valid_mask]
        gene_names = gene_names[valid_mask]

    # H/M/L 标签函数（只用于 token 标记，不是 label）
    if label_mode == "percentile":
        low, high = np.percentile(X, [33, 66])
        def label_func(vals):
            return np.where(vals > high, "(H)", np.where(vals < low, "(L)", "(M)"))
    elif label_mode == "zscore":
        mean, std = np.mean(X), np.std(X)
        def label_func(vals):
            z = (vals - mean) / std
            return np.where(z > 1, "(H)", np.where(z < -1, "(L)", "(M)"))
    else:  # threshold
        def label_func(vals):
            return np.where(vals >= 3.0, "(H)", np.where(vals < 2.0, "(L)", "(M)"))

    # 每个 spot 按表达值从大到小选 top_k 基因并附标签
    tokens_list = []
    for i in range(X.shape[0]):
        expr = X[i, :]
        top_idx = np.argsort(-expr)[:top_k]
        top_genes = gene_names[top_idx]
        top_exprs = expr[top_idx]
        top_tags = label_func(top_exprs)
        top_tokens = [f"{g}{t}" for g, t in zip(top_genes, top_tags)]
        tokens_list.append(top_tokens)

    return tokens_list

# ---------------------------
# 从 HE 图像提取 32x32 patch
# ---------------------------
def extract_spot_patches_exact32(adata, image_path, channel_first=True, normalize=True):
    img = Image.open(image_path).convert("RGB")
    spatial_coords = adata.obsm["spatial"]
    patch_size = 32
    patches = []

    for i in range(spatial_coords.shape[0]):
        x, y = spatial_coords[i]
        left   = int(x - patch_size / 2)
        upper  = int(y - patch_size / 2)
        right  = left + patch_size
        lower  = upper + patch_size

        patch = img.crop((left, upper, right, lower)) 
        patch_np = np.array(patch).astype(np.float32)

        if normalize:
            patch_np /= 255.0  # → [0,1]

        if channel_first:
            patch_np = patch_np.transpose(2, 0, 1)  # → [C,32,32]

        patches.append(patch_np)

    return np.stack(patches)  # → shape: [N, C, 32, 32]

# ---------------------------
# 处理一个 count_df（检测是否鼠类并映射）
# ---------------------------
def preprocess_gene_matrix(count_df):
    """
    处理 count_df：
    - 如果是小鼠 → 仅保留能映射到人类的基因，否则返回 None
    - 如果是人类 → 全部大写
    """
    gene_names = count_df.columns.tolist()

    # 判断是否是小鼠（看第一个基因是否包含小写）
    if any(c.islower() for c in gene_names[0]):
        # 小写化全部列名
        gene_names_lower = [g.lower() for g in gene_names]

        # 只保留能映射的鼠基因
        available_mouse_genes = [g for g in gene_names_lower if g in mouse2human]

        if len(available_mouse_genes) == 0:
            # ❌ 没有能映射的鼠基因 → 返回 None，让外层跳过
            return None

        # 只保留能映射的列
        count_df = count_df.loc[:, [g for g in gene_names if g.lower() in available_mouse_genes]]

        # 映射到人类基因名
        mapped_names = [mouse2human[g.lower()] for g in count_df.columns]
        # 过滤掉映射后为空的
        keep_cols, keep_names = [], []
        for col, new_name in zip(count_df.columns, mapped_names):
            if new_name and isinstance(new_name, str) and len(new_name.strip()) > 0:
                keep_cols.append(col)
                keep_names.append(new_name)
        count_df = count_df.loc[:, keep_cols]
        count_df.columns = keep_names

        if count_df.shape[1] == 0:
            # 如果过滤后没有列，直接返回 None 跳过
            return None

    else:
        # 人类 → 大写
        count_df.columns = [g.upper() for g in gene_names]

    return count_df



# ---------------------------
# 核心批量处理函数
# ---------------------------
def process_all_samples(base_dir, output_dir):
    coord_dir = os.path.join(base_dir, "coord")
    gene_dir = os.path.join(base_dir, "gene_exp")
    img_dir = os.path.join(base_dir, "image")

    coord_files = glob.glob(os.path.join(coord_dir, "*_coord.csv"))
    sample_prefixes = [os.path.basename(f).replace("_coord.csv", "") for f in coord_files]

    os.makedirs(output_dir, exist_ok=True)

    for prefix in tqdm(sample_prefixes, desc="Processing samples"):
        save_path = os.path.join(output_dir, f"{prefix}.npz")
        
        # 如果 npz 已存在 → 跳过
        if os.path.exists(save_path):
            continue

        coord_path = os.path.join(coord_dir, f"{prefix}_coord.csv")
        gene_path  = os.path.join(gene_dir,  f"{prefix}_count.csv")
        img_path   = os.path.join(img_dir,   f"{prefix}.png")

        # 检查文件是否齐全
        if not (os.path.exists(coord_path) and os.path.exists(gene_path) and os.path.exists(img_path)):
            print(f"⚠️ 缺少文件，跳过 {prefix}")
            continue

        # ---- 1. 读取 CSV ----
        count_df = pd.read_csv(gene_path, index_col=0)
        coord_df = pd.read_csv(coord_path, index_col=0)

        # ---- 1.5 鼠类 → 转人类基因 ----
        count_df = preprocess_gene_matrix(count_df)
        if count_df is None or count_df.shape[1] == 0:
            # ❌ 没有任何基因可用 → 跳过
            print(f"⚠️ 无法映射，跳过 {prefix}")
            continue

        # ---- 2. 构造 AnnData ----
        adata = ad.AnnData(
            X = count_df.values,
            obs = pd.DataFrame(index = count_df.index),
            var = pd.DataFrame(index = count_df.columns)
        )
        adata.obsm["spatial"] = coord_df.loc[adata.obs.index, ["xaxis", "yaxis"]].values

        # ---- 3. 标准预处理 ----
        sc.pp.filter_cells(adata, min_genes=200)
        sc.pp.filter_genes(adata, min_cells=3)
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # ---- 4. 生成 tokens ----
        tokens = tokenize_spots(
            ad_ge=adata,
            top_k=2048,
            label_mode="threshold",
            valid_gene_names=valid_genes
        )

        # ---- 5. 提取 patch ----
        patches = extract_spot_patches_exact32(adata, img_path)

        # ---- 6. 保存 npz ----
        np.savez_compressed(
            save_path,
            patch=patches,
            tokens=np.array(tokens, dtype=object)
        )


In [14]:
process_all_samples(
    base_dir="./STimage-1K4M/Visium",        # 输入数据根目录
    output_dir="./npzdata"   # 每个样本单独保存 npz
)

Processing samples:   0%|          | 0/994 [00:00<?, ?it/s]

Processing samples:   7%|▋         | 67/994 [01:15<17:26,  1.13s/it]

⚠️ 无法映射，跳过 GSE178221_GSM5384657


Processing samples:   7%|▋         | 68/994 [03:26<59:22,  3.85s/it]

⚠️ 无法映射，跳过 GSE178221_GSM5384658


...

Processing samples:  72%|███████▏  | 715/994 [8:19:29<5:28:28, 70.64s/it] /home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
Processing samples:  72%|███████▏  | 7

⚠️ 无法映射，跳过 GSE242270_GSM7757568


Processing samples:  73%|███████▎  | 721/994 [8:23:49<4:27:33, 58.80s/it]

⚠️ 无法映射，跳过 GSE242270_GSM7757569


Processing samples:  76%|███████▌  | 755/994 [8:42:57<3:58:33, 59.89s/it]/home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/mwc/miniconda3/envs/st/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
Processing samples:  76%|███████▌  | 75